# SIFT + LightGlue ORB-SLAM3 fork -- KITTI benchmark (Kaggle GPU)

Runs `orbslam3_sift_kitti_ate` (the SIFT-feature fork of ORB-SLAM3 with LightGlue as the frame-to-frame matcher, see `DEBUGGING.md` parts 56-58) against a KITTI odometry sequence, using a Kaggle GPU so LightGlue's O(N^2) attention runs at its intended speed instead of the ~9s/pair this project measured on CPU locally.

**Before running:**
1. Notebook settings -> Accelerator -> GPU (T4 x2 or P100). Internet -> On.
2. Add Data -> attach a KITTI odometry (grayscale, sequence 00) dataset, or upload one -- needs `.../image_0/*.png` and a `poses/00.txt`-style ground-truth file somewhere under `/kaggle/input`.
3. Set `REPO_URL` below to this project's GitHub URL (must be pushed first -- this notebook only clones over HTTPS, so the repo needs to be public, or you'll need to bake a token into the URL yourself).

The heavy one-time setup (installing apt packages, building g2o from ORB-SLAM3's own `Thirdparty/g2o`, downloading the ONNX Runtime GPU release) is cached under `kaggle/g2o_build` and `kaggle/onnxruntime` for the rest of the session -- rerunning the run cell after tweaking `START_FRAME`/`MAX_FRAMES` does not repeat it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")

In [ ]:
# One-time setup: apt deps, g2o build, ONNX Runtime GPU download, cmake
# configure + build. Safe to rerun (skips steps whose output already exists).
!cd {REPO_DIR} && bash kaggle/setup_and_run.sh

The cell above both builds AND runs a default 0-1000-frame benchmark (env var defaults in `kaggle/setup_and_run.sh`). To rerun with different frame ranges without repeating the build, use the cell below instead.

In [ ]:
import os

env = os.environ.copy()
env["SKIP_BUILD"] = "1"
env["START_FRAME"] = "0"
env["MAX_FRAMES"] = "1000"
env["OUT_PREFIX"] = "/kaggle/working/lightglue_1_1000"
# Uncomment and set explicitly if auto-detection under /kaggle/input picks the wrong dataset:
# env["KITTI_SEQ_DIR"] = "/kaggle/input/<your-dataset>/sequences/00"
# env["KITTI_POSES"] = "/kaggle/input/<your-dataset>/poses/00.txt"

import subprocess, sys
result = subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env)
sys.exit(result.returncode) if result.returncode != 0 else None

In [ ]:
# Fragment-by-fragment ATE report and the [sbp-diag] match-quality summary
# are printed directly to stdout by the run cell above (search for
# "[atlas-coverage]" / "[fragment" / "[sbp-diag]"). The keyframe trajectory
# itself is written to OUT_PREFIX + "_KeyFrameTrajectory.txt":
!tail -n 20 /kaggle/working/lightglue_1_1000_KeyFrameTrajectory.txt